<a href="https://colab.research.google.com/github/ford442/the_jokesters/blob/main/utils/convert_kimi_vl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Unified FP32 ONNX Export: Vicuna-7B + Kimi-VL-A3B (A100)**Purpose**: Export both text-only (Vicuna) and vision (Kimi-VL) models to FP32 ONNX for WebGPU compatibility**Target**: ONNX Runtime Web with WebGPU EP  **Runtime**: A100 GPU (40GB) — FP32 requires more VRAM than FP16⚠️ **Critical for WebGPU**: FP32 is universally supported; FP16 requires `shader-f16` extension

In [ ]:
# CELL 1: Environment Setup (A100 Required)
!pip install -q --upgrade pip
!pip install -q transformers==4.51.2 accelerate optimum[exporters,onnxruntime-gpu] onnx onnxruntime-gpu

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print("✅ A100 FP32 Setup Complete")

In [ ]:
# CELL 2: Export Vicuna 7B → FP32 ONNX (for ModelRouter)
!optimum-cli export onnx \
  --model lmsys/vicuna-7b-v1.5 \
  --task text-generation-with-past \
  --device cuda \
  --optimize O2 \
  --no-fp16 \
  --output /content/vicuna_7b_onnx_fp32

print("✅ Vicuna 7B FP32 exported")

In [ ]:
# CELL 3: Export Kimi-VL → FP32 ONNX (Fixed from FP16)
from optimum.exporters.onnx import main_export

main_export(
    model_name_or_path="moonshotai/Kimi-VL-A3B-Thinking-2506",
    output="/content/kimi_vl_onnx_fp32",
    task="image-to-text",
    trust_remote_code=True,
    device="cuda",
    no_fp16=True,  # CRITICAL: Force FP32 for WebGPU compatibility
    optimize="O2",
    opset=17,
    use_external_data_format=True,  # Shard for browser
    batch_size=1,
)

print("✅ Kimi-VL FP32 exported")

In [ ]:
# CELL 4: Convert to Web-Ready Format (512MB Shards)
import onnx
import os

def convert_to_web_format(input_path, output_dir, shard_mb=512):
    """Convert ONNX to FP32 web format with external data"""
    os.makedirs(output_dir, exist_ok=True)
    
    model = onnx.load(input_path)
    
    # Force FP32 if any FP16 snuck in
    for tensor in model.graph.initializer:
        if tensor.data_type == 10:  # FP16
            print(f"Converting {tensor.name} from FP16 to FP32")
            tensor.data_type = 1  # FP32
    
    onnx.save_model(
        model,
        f"{output_dir}/model.onnx",
        save_as_external_data=True,
        location="weights",
        size_threshold=shard_mb * 1024 * 1024,
        convert_attribute=True
    )
    
    # List the shards
    shards = [f for f in os.listdir(output_dir) if f.startswith('weights')]
    print(f"✅ Created {len(shards)} weight shards in {output_dir}")
    return f"{output_dir}/model.onnx"

# Convert both
convert_to_web_format("/content/vicuna_7b_onnx_fp32/model.onnx", "/content/web_vicuna")
convert_to_web_format("/content/kimi_vl_onnx_fp32/model.onnx", "/content/web_kimi")

In [ ]:
# CELL 5: Upload to HuggingFace
from huggingface_hub import HfApi, create_repo

api = HfApi()

# Create repos (uncomment if needed)
# create_repo("ford442/vicuna-7b-webgpu", exist_ok=True)
# create_repo("ford442/kimi-vl-webgpu", exist_ok=True)

print("⬆️ Uploading Vicuna...")
api.upload_folder(
    folder_path="/content/web_vicuna",
    repo_id="ford442/vicuna-7b-webgpu",
    repo_type="model"
)

print("⬆️ Uploading Kimi-VL...")
api.upload_folder(
    folder_path="/content/web_kimi",
    repo_id="ford442/kimi-vl-webgpu",
    repo_type="model"
)

print("✅ Both models uploaded!")